# HIV Surveillance Data Simulator

**Purpose:** Generate 5,600 simulated HIV case records with realistic data quality issues for QA testing

**Output:** `simulated_data.csv` with ~34% records flagged, 66% clean | ⚠️ Simulated data only

In [9]:
# import the packages to create the data
import pandas as pd
import numpy as np
from faker import Faker
from datetime import timedelta
import random


In [10]:
# initialize the faker to make fake data
fake = Faker()
Faker.seed(42)
np.random.seed(42)


In [11]:
# This defines the categories that exist in the invented dataset for each of these variables
sex_at_birth = ["Male", "Female"]
gender = ["Male", "Female", "Transgender", "Non-binary"]
race_ethnicity = ["White", "Black", "Hispanic", "Asian", "Native American", "Other"]
transmission = ["MSM", "IDU", "Heterosexual", "MSM/IDU", "Perinatal", "Unknown"]
counties = [
    "Boone", "Cole", "Jackson", "Greene", "St. Louis City", "St. Louis County",
    "Clay", "Platte", "Cass", "Buchanan", "Jasper", "Cape Girardeau",
    "Christian", "Taney", "Phelps", "Callaway", "Audrain", "Howard",
    "Moniteau", "Osage", "Lafayette", "Johnson", "Pettis", "Saline",
    "Benton", "Miller", "Pulaski", "Texas", "Dent", "Laclede"
]
record_status = ["Confirmed", "Probable"]



In [12]:
# Here I weight the county variables for realistic simulation of data
# County weights (urban counties dominate)
county_weights = {
    "St. Louis City": 0.18,
    "St. Louis County": 0.22,
    "Jackson": 0.20,
    "Greene": 0.08,
    "Boone": 0.06,
    "Clay": 0.04,
    "Platte": 0.03,
    "Cass": 0.03,
    "Cole": 0.03,
    "Jasper": 0.03,
    "Other": 0.07
}

# Expand "Other" into remaining counties
other_counties = [
    "Buchanan", "Cape Girardeau", "Christian", "Taney", "Phelps",
    "Callaway", "Audrain", "Howard", "Moniteau", "Osage", "Lafayette",
    "Johnson", "Pettis", "Saline", "Benton", "Miller", "Pulaski",
    "Texas", "Dent", "Laclede"
]

county_choices = (
    ["St. Louis City", "St. Louis County", "Jackson", "Greene", "Boone",
     "Clay", "Platte", "Cass", "Cole", "Jasper"] +
    other_counties
)

county_probs = (
    [county_weights[c] for c in county_weights if c != "Other"] +
    [county_weights["Other"] / len(other_counties)] * len(other_counties)
)


In [13]:
# Here I weight the other variables
sex_weights = [0.78, 0.22]  # Male, Female

gender_probs = [0.75, 0.20, 0.03, 0.02]  # Male, Female, Trans, Non-binary

race_probs = [0.45, 0.32, 0.15, 0.05, 0.02, 0.01]  # White, Black, Hispanic, Asian, Native, Other

transmission_probs = [0.55, 0.15, 0.15, 0.05, 0.02, 0.08]  # MSM dominant


In [14]:
# this is so that the code later has gender being probabalistically related to sex
transmission_probs_by_sex = {
    "Male": {
        "MSM": 0.65,
        "Heterosexual": 0.15,
        "IDU": 0.12,
        "MSM/IDU": 0.05,
        "Perinatal": 0.01,
        "Unknown": 0.02
    },
    "Female": {
        "Heterosexual": 0.65,
        "IDU": 0.18,
        "MSM": 0.03,          # rare but possible
        "MSM/IDU": 0.01,
        "Perinatal": 0.10,
        "Unknown": 0.03
    }
}


In [19]:
def generate_case(case_id):
    # FIX: Age calculation
    hiv_dx = fake.date_between(start_date='-20y', end_date='today')
    age_at_diagnosis = random.randint(18, 80)
    dob = hiv_dx - timedelta(days=int(age_at_diagnosis * 365.25))
    
    # REDUCE: AIDS before HIV (2% error rate instead of ~15%)
    rand_aids = random.random()
    if rand_aids < 0.6:
        aids_dx = None
    elif rand_aids < 0.62:  # 2% have date swap error
        aids_dx = hiv_dx + timedelta(days=random.randint(-365, -30))
    else:  # 38% progress to AIDS normally
        aids_dx = hiv_dx + timedelta(days=random.randint(180, 2000))
    
    # REDUCE: Report date errors (3% instead of ~20%)
    if random.random() < 0.03:
        report_date = hiv_dx + timedelta(days=random.randint(-200, 30))
    else:
        report_date = hiv_dx + timedelta(days=random.randint(1, 365))
    
    sex = random.choices(sex_at_birth, weights=sex_weights, k=1)[0]
    
    tx_dist = transmission_probs_by_sex[sex]
    transmission_category = random.choices(
        population=list(tx_dist.keys()),
        weights=list(tx_dist.values()),
        k=1
    )[0]
    
    # KEEP: 5% missing transmission
    if random.random() < 0.05:
        transmission_category = None
    
    # REDUCE: Negative lab values (5% instead of 33%)
    cd4_options = [random.randint(20, 1500)] * 85 + [None] * 10 + [-10] * 5
    vl_options = [random.randint(20, 1_000_000)] * 75 + [0] * 20 + [-5] * 5
    
    # REDUCE: Future lab dates (3% instead of ~30%)
    if random.random() < 0.03:
        last_lab_date = hiv_dx + timedelta(days=random.randint(30, 3650))
    else:
        last_lab_date = hiv_dx + timedelta(days=random.randint(30, 1095))
    
    return {
        "case_id": case_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "dob": dob,
        "sex_at_birth": sex,
        "current_gender": random.choices(gender, weights=gender_probs, k=1)[0],
        "race_ethnicity": random.choices(race_ethnicity + [None], weights=race_probs + [0.05], k=1)[0],
        "county": random.choices(county_choices, weights=county_probs, k=1)[0],
        "hiv_diagnosis_date": hiv_dx,
        "aids_diagnosis_date": aids_dx,
        "transmission_category": transmission_category,
        "last_cd4": random.choice(cd4_options),
        "last_viral_load": random.choice(vl_options),
        "last_lab_date": last_lab_date,
        "report_date": report_date,
        "record_status": random.choice(record_status)
    }

In [20]:
# a for loop that creates 5000 cases and assigns them to a dataframe
base_cases = [generate_case(i) for i in range(1, 5001)]
df = pd.DataFrame(base_cases)


In [21]:
# this creates duplicates at around 12%
dup_fraction = 0.12
dupes = df.sample(frac=dup_fraction).copy()

dupes["case_id"] += 10000

# Introduce common duplicate inconsistencies
dupes["last_name"] = dupes["last_name"].apply(
    lambda x: x[:-1] if len(x) > 3 else x
)

dupes["county"] = dupes["county"].sample(frac=1).values
dupes["record_status"] = "Duplicate"

df = pd.concat([df, dupes], ignore_index=True)


In [22]:
df

,case_id,first_name,last_name,dob,sex_at_birth,current_gender,race_ethnicity,county,hiv_diagnosis_date,aids_diagnosis_date,transmission_category,last_cd4,last_viral_load,last_lab_date,report_date,record_status
0,1,Christian,Rodriguez,1939-11-05,Male,Male,Black,St. Louis County,2007-11-05,2010-08-27,MSM,206.0,668588,2010-01-07,2008-11-01,Confirmed
1,2,Michelle,Arias,1941-06-02,Female,Male,White,St. Louis County,2013-06-02,2015-12-04,Heterosexual,1229.0,69208,2013-10-18,2014-01-09,Probable
2,3,Tammy,Pena,1980-12-04,Female,Male,Hispanic,St. Louis County,2017-12-04,2021-11-14,Heterosexual,NaN,310752,2018-10-03,2018-05-10,Probable
3,4,Scott,Zamora,1971-12-15,Male,Male,White,St. Louis County,2014-12-14,None,MSM,-10.0,337930,2015-03-10,2015-03-19,Probable
4,5,Antonio,Campbell,1944-09-05,Female,Male,Black,Boone,2017-09-05,None,IDU,518.0,-5,2018-01-06,2017-09-07,Probable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5595,10374,Michael,Cabrer,1990-08-24,Female,Female,Black,St. Louis County,2021-08-23,2026-04-03,MSM,1370.0,246397,2021-12-14,2022-05-03,Duplicate
5596,11853,Ashley,Turne,1972-11-25,Male,Male,Black,Osage,2013-11-25,2015-01-09,MSM,882.0,535104,2016-10-08,2014-03-31,Duplicate
5597,14083,George,Coope,1963-06-05,Female,Male,Hispanic,St. Louis City,2010-06-04,None,Perinatal,1292.0,875166,2011-10-10,2010-08-25,Duplicate
5598,13650,David,Jone,1932-10-31,Male,Male,White,Greene,2010-10-31,2014-11-29,MSM,302.0,0,2013-01-12,2011-02-18,Duplicate


In [23]:
# save the simulated data to a csv file
df = df.sample(frac=1).reset_index(drop=True)
df.to_csv("simulated_data.csv", index=False)
